In [1]:
!pip install -q dspy-ai requests beautifulsoup4 pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.2/421.2 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.2/290.2 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.5/23.5 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.3/278.3 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 6.8 MB/s eta 0:00:00


In [2]:
import os
import csv
import time
import requests
from bs4 import BeautifulSoup
from typing import List
from pydantic import BaseModel, Field
import dspy

In [3]:

LONGCAT_API_KEY = os.environ.get("LONGCAT_API_KEY", "ak_2k153H99M3mp48409f95w22L7xM5B")
LONGCAT_BASE_URL = "https://api.longcat.chat/openai/v1"
LONGCAT_MODEL = "openai/LongCat-2.0"

lm = dspy.LM(
    model=LONGCAT_MODEL,
    api_key=LONGCAT_API_KEY,
    api_base=LONGCAT_BASE_URL,
)
dspy.configure(lm=lm)

In [4]:

class EntityWithAttr(BaseModel):
    entity: str = Field(description="the named entity, exact string as it appears in text")
    attr_type: str = Field(description="semantic type, e.g. Crop, Process, Measurement, Disease, Drug, Concept, Organization")


class ExtractEntities(dspy.Signature):
    """Extract the key named entities from the paragraph, with their semantic type."""
    paragraph: str = dspy.InputField()
    entities: List[EntityWithAttr] = dspy.OutputField()


class ExtractTriples(dspy.Signature):
    """Extract subject-predicate-object relationship triples between the given entities,
    based strictly on what the paragraph states. Only use entities from the provided list."""
    paragraph: str = dspy.InputField()
    entities: List[str] = dspy.InputField()
    triples: List[str] = dspy.OutputField(desc="each triple formatted as 'subject|predicate|object'")


class DedupEntities(dspy.Signature):
    """Given a list of entity strings that may contain near-duplicates (different phrasing,
    casing, or abbreviations of the same real-world thing), merge them into a deduplicated
    list, choosing the clearest, most complete canonical form for each distinct entity."""
    items: List[str] = dspy.InputField()
    deduplicated: List[str] = dspy.OutputField()
    confidence: float = dspy.OutputField(desc="confidence 0.0-1.0 that the dedup is correct")


extractor = dspy.Predict(ExtractEntities)
triple_extractor = dspy.Predict(ExtractTriples)
dedup_predictor = dspy.Predict(DedupEntities)

In [5]:

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; AssignmentBot/1.0; +https://example.com)"}


def scrape_text(url: str, max_chars: int = 6000) -> str:
    """Best-effort scrape. Publisher sites (Nature, ScienceDirect, NCBI, Medscape) often
    block simple requests or gate content behind JS/paywalls — in that case this returns
    empty string and the URL is logged as skipped; note that in your write-up."""
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
            tag.decompose()
        text = " ".join(soup.get_text(separator=" ").split())
        return text[:max_chars]
    except Exception as e:
        print(f"  [WARN] could not scrape {url}: {e}")
        return ""

In [6]:

def deduplicate_with_lm(items: List[str], batch_size: int = 10, target_confidence: float = 0.9, max_tries: int = 3) -> List[str]:
    if not items:
        return []
    deduped = []
    for i in range(0, len(items), batch_size):
        batch = items[i:i + batch_size]
        pred = None
        for _ in range(max_tries):
            pred = dedup_predictor(items=batch)
            if pred.confidence >= target_confidence:
                break
        deduped.extend(pred.deduplicated)
    if len(deduped) > batch_size:
        pred = dedup_predictor(items=deduped)
        return list(dict.fromkeys(pred.deduplicated))
    return list(dict.fromkeys(deduped))

In [7]:

def _clean(s: str) -> str:
    return s.strip().replace('"', "'").replace("\n", " ")[:60]


def triples_to_mermaid(triples: List[str], entity_list: List[str]) -> str:
    entity_set = {e.strip().lower() for e in entity_list}
    lines = ["graph TD"]
    seen_edges = set()
    for t in triples:
        parts = t.split("|")
        if len(parts) != 3:
            continue
        src, lbl, dst = (p.strip() for p in parts)
        if src.lower() not in entity_set or dst.lower() not in entity_set:
            continue
        lbl = lbl[:40]
        edge = (src, lbl, dst)
        if edge in seen_edges:
            continue
        seen_edges.add(edge)
        lines.append(f'    {_clean(src)} -- "{lbl}" --> {_clean(dst)}')
    if len(lines) == 1:
        lines.append("    NoData[No valid relationships extracted]")
    return "\n".join(lines)

In [8]:

URLS = [
    "https://en.wikipedia.org/wiki/Sustainable_agriculture",
    "https://www.nature.com/articles/d41586-025-03353-5",
    "https://www.sciencedirect.com/science/article/pii/S1043661820315152",
    "https://www.ncbi.nlm.nih.gov/pmc/articles/PMC10457221/",
    "https://www.fao.org/3/y4671e/y4671e06.htm",
    "https://www.medscape.com/viewarticle/time-reconsider-tramadol-chronic-pain2025a1000ria",
    "https://www.sciencedirect.com/science/article/pii/S0378378220307088",
    "https://www.frontiersin.org/news/2025/09/01/rectangle-telescope-finding-habitable-planets",
    "https://www.medscape.com/viewarticle/second-dose-boosts-shingles-protection-adults-aged-65-years-2025a1000ro7",
    "https://www.theguardian.com/global-development/2025/oct/13/astro-ambassadors-stargazers-himalayas-hanle-ladakh-india",
]


def process_url(url: str, idx: int, csv_writer) -> None:
    print(f"\n=== [{idx}] {url} ===")
    text = scrape_text(url)
    if not text:
        print("  -> skipped (scrape blocked/failed)")

        with open(f"mermaid_{idx}.md", "w") as f:
            f.write("```mermaid\ngraph TD\n    NoData[Could not scrape source]\n```\n")
        return


    chunks = [text[i:i + 1500] for i in range(0, len(text), 1500)][:3]

    raw_entities = []
    for chunk in chunks:
        try:
            pred = extractor(paragraph=chunk)
            raw_entities.extend([(e.entity, e.attr_type) for e in pred.entities])
        except Exception as e:
            print(f"  [WARN] extraction failed on a chunk: {e}")

    if not raw_entities:
        print("  -> no entities extracted")
        with open(f"mermaid_{idx}.md", "w") as f:
            f.write("```mermaid\ngraph TD\n    NoData[No entities extracted]\n```\n")
        return

    entity_strings = [e[0] for e in raw_entities]
    deduped_entities = deduplicate_with_lm(entity_strings)


    type_map = {ent.lower(): etype for ent, etype in raw_entities}
    seen_for_url = set()
    for dedup_ent in deduped_entities:
        if dedup_ent.lower() in seen_for_url:
            continue
        seen_for_url.add(dedup_ent.lower())
        etype = type_map.get(dedup_ent.lower(), "Unknown")
        csv_writer.writerow([url, dedup_ent, etype])

    try:
        tpred = triple_extractor(paragraph=" ".join(chunks), entities=deduped_entities)
        triples = tpred.triples
    except Exception as e:
        print(f"  [WARN] triple extraction failed: {e}")
        triples = []

    mermaid_code = triples_to_mermaid(triples, deduped_entities)
    with open(f"mermaid_{idx}.md", "w") as f:
        f.write(f"```mermaid\n{mermaid_code}\n```\n")
    print(f"  -> {len(deduped_entities)} entities, mermaid_{idx}.md saved")


def main():
    with open("tags.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["link", "tag", "tag_type"])
        for i, url in enumerate(URLS, start=1):
            process_url(url, i, writer)
            time.sleep(2)
    print("\nDone. Outputs: mermaid_1.md .. mermaid_10.md, tags.csv")


if __name__ == "__main__":
    main()


=== [1] https://en.wikipedia.org/wiki/Sustainable_agriculture ===
  [WARN] extraction failed on a chunk: [LongCat-2.0] litellm.RateLimitError: RateLimitError: OpenAIException - 调用失败：Token 额度不足。欢迎反馈模型使用case（https://longcat.chat/platform/feedback）获取更多额度
  [WARN] extraction failed on a chunk: [LongCat-2.0] litellm.RateLimitError: RateLimitError: OpenAIException - 调用失败：Token 额度不足。欢迎反馈模型使用case（https://longcat.chat/platform/feedback）获取更多额度
  [WARN] extraction failed on a chunk: [LongCat-2.0] litellm.RateLimitError: RateLimitError: OpenAIException - 调用失败：Token 额度不足。欢迎反馈模型使用case（https://longcat.chat/platform/feedback）获取更多额度
  -> no entities extracted

=== [2] https://www.nature.com/articles/d41586-025-03353-5 ===
  [WARN] extraction failed on a chunk: [LongCat-2.0] litellm.RateLimitError: RateLimitError: OpenAIException - 调用失败：Token 额度不足。欢迎反馈模型使用case（https://longcat.chat/platform/feedback）获取更多额度
  [WARN] extraction failed on a chunk: [LongCat-2.0] litellm.RateLimitError: RateLimitError: OpenA

In [9]:
main()


=== [1] https://en.wikipedia.org/wiki/Sustainable_agriculture ===
  [WARN] extraction failed on a chunk: [LongCat-2.0] litellm.RateLimitError: RateLimitError: OpenAIException - 调用失败：Token 额度不足。欢迎反馈模型使用case（https://longcat.chat/platform/feedback）获取更多额度
  [WARN] extraction failed on a chunk: [LongCat-2.0] litellm.RateLimitError: RateLimitError: OpenAIException - 调用失败：Token 额度不足。欢迎反馈模型使用case（https://longcat.chat/platform/feedback）获取更多额度
  [WARN] extraction failed on a chunk: [LongCat-2.0] litellm.RateLimitError: RateLimitError: OpenAIException - 调用失败：Token 额度不足。欢迎反馈模型使用case（https://longcat.chat/platform/feedback）获取更多额度
  -> no entities extracted

=== [2] https://www.nature.com/articles/d41586-025-03353-5 ===
  [WARN] extraction failed on a chunk: [LongCat-2.0] litellm.RateLimitError: RateLimitError: OpenAIException - 调用失败：Token 额度不足。欢迎反馈模型使用case（https://longcat.chat/platform/feedback）获取更多额度
  [WARN] extraction failed on a chunk: [LongCat-2.0] litellm.RateLimitError: RateLimitError: OpenA

## Check outputs
Download `tags.csv` and the `mermaid_*.md` files from the Colab file browser.

In [10]:
import os
print(sorted(f for f in os.listdir('.') if f.startswith(('mermaid_','tags'))))

['mermaid_1.md', 'mermaid_10.md', 'mermaid_2.md', 'mermaid_3.md', 'mermaid_4.md', 'mermaid_5.md', 'mermaid_6.md', 'mermaid_7.md', 'mermaid_8.md', 'mermaid_9.md', 'tags.csv']


In [12]:
import pandas as pd
df = pd.read_csv("tags.csv")
print(df.shape)
df.head(20)

(0, 3)


,link,tag,tag_type
